# 02 — Experimento de Fine-Tuning

Entrenamiento del modelo con LoRA/QLoRA sobre el dataset de iTimeControl.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from src.fine_tuning.config import load_training_config

cfg = load_training_config('../config.yaml')
print(f'Modelo base: {cfg.base_model}')
print(f'Épocas: {cfg.num_epochs}')
print(f'LR: {cfg.learning_rate}')
print(f'LoRA r: {cfg.lora.r}')

In [ ]:
# Verificar que el dataset está listo
from pathlib import Path
from src.utils.helpers import load_config

config = load_config('../config.yaml')
datasets_dir = Path('../') / config['paths']['datasets_dir']
fmt = config['dataset']['format']

train_file = datasets_dir / f'train_{fmt}.jsonl'
val_file   = datasets_dir / f'val_{fmt}.jsonl'

print(f'Train: {train_file} - existe: {train_file.exists()}')
print(f'Val:   {val_file} - existe: {val_file.exists()}')

In [ ]:
# Iniciar entrenamiento (ejecutar solo cuando todo esté listo)
from src.fine_tuning.trainer import train

# ADVERTENCIA: Esto requiere GPU y puede tomar varias horas
# Descomenta la siguiente línea cuando estés listo:
# trainer = train(cfg)

In [ ]:
# Visualizar curva de pérdida (después del entrenamiento)
import json
import matplotlib.pyplot as plt
from pathlib import Path

log_file = Path('../models/checkpoints/trainer_state.json')

if log_file.exists():
    state = json.loads(log_file.read_text())
    history = state.get('log_history', [])
    
    train_losses = [(h['step'], h['loss']) for h in history if 'loss' in h]
    eval_losses  = [(h['step'], h['eval_loss']) for h in history if 'eval_loss' in h]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    if train_losses:
        steps, losses = zip(*train_losses)
        ax.plot(steps, losses, label='Train Loss', color='steelblue')
    if eval_losses:
        steps, losses = zip(*eval_losses)
        ax.plot(steps, losses, label='Val Loss', color='tomato')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Curva de pérdida — Fine-Tuning iTimeControl')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Entrena el modelo primero.')